# Projeto de Controladores

In [36]:
# Bibliotecas utilizadas
import matplotlib.pyplot as plt
import sympy as sp
from sympy import symbols, I, Abs, re, im, Poly
from sympy.physics.control import *
import numpy as np
import matplotlib.pyplot as plt
import math
import cmath
from collections import Counter
#import plotly.graph_objects as go

In [37]:
# Símbolos utilizados
s, k = sp.symbols('s k')

## Funções auxiliares

In [38]:
def polinomio(entrada):
    '''
    Função que transforma coeficientes em polinômios
    '''
    coeficientes = entrada.strip().split()  # Transforma a entrada em uma lista de strings e remove espaços extras
    coeficientes.reverse() # Inverte a lista
    lista_polinomio = []

    # Percorre de trás para frente para começar pelo maior expoente
    for exp in range(len(coeficientes) - 1, -1, -1):
        coef = coeficientes[exp]

        # Pula coeficientes que são zero
        if coef == "0":
            continue

        # Formata o termo dependendo do expoente
        if exp > 1:
            termo = f"{coef}*s**{exp}"
        elif exp == 1:
            termo = f"{coef}*s"
        else:
            termo = f"{coef}"

        lista_polinomio.append(termo)

    return " + ".join(lista_polinomio)


In [39]:
def definir_controlador():
    PI = False
    PD = False
    PID = False
    controlador = ""

    while(controlador != "PI" and controlador != "PD" and controlador != "PID"):
        controlador = input("Forneça o tipo de controlador").upper()
        if(controlador == "PI"):
            PI = True
            print("Controlador PI")
        elif(controlador == "PD"):
            PD = True
            print("Controlador PD")
        elif(controlador == "PID"):
            PID = True
            print("Controlador PID")
    return PI,PD,PID


In [40]:
def definir_problema():
    metodo_1 = False
    metodo_2 = False
    metodo_3 = False
    print("Digite 1 para Mp e Epsilon.")
    print("Digite 2 para fator de amortecimento e frequência natural de amortecimento.")
    print("Digite 3 para localização de polos de malha")
    especificacoes = 0
    while(especificacoes < 1 or especificacoes > 3):
        especificacoes = int(input("Digite o número: "))
        if(especificacoes == 1):
            metodo_1 = True
        elif(especificacoes == 2):
            metodo_2 = True
        elif(especificacoes == 3):
            metodo_3 = True
    return metodo_1, metodo_2, metodo_3


In [51]:
def definir_polos_dominantes(metodo1, metodo2, metodo3):
    if(metodo1):
        Mp = int(input("Digite o Mp máximo (%): "))
        print(f"Mp = {Mp}%")
        t_acomodacao = int(input("Digite o tempo de acomodação (s):"))
        tol = int(input("Digite 2 ou 5 para a tolerância (%): "))
        while(tol != 2 and tol != 5):
            tol = int(input("Digite 2 ou 5 para a tolerância (%): "))
        
        print(f"Acomodação {t_acomodacao}s ({tol}%)")

        # Cálculo do epsilon e frequência natural
        epsilon = math.sqrt(math.log(Mp/100)**2/(math.pi**2 + math.log(Mp/100)**2))
        if(tol == 2):
            w_n = 4/(t_acomodacao*epsilon)
        elif(tol == 5):
            w_n = 3/(t_acomodacao*epsilon)

        w_d = w_n*math.sqrt(1 - epsilon**2)
        sigma = epsilon*w_n

        polo_1 = -sigma + w_d * I
        polo_2 = -sigma - w_d  * I

    if(metodo2):
        epsilon = int(input("Digite o valor do epsilon: "))
        w_n = int(input("Digite o valor da frequência natural de oscilação: "))
        w_d = w_n*math.sqrt(1 - epsilon**2)
        polo_1 = -sigma + w_d * I
        polo_2 = -sigma - w_d * I
    if(metodo3):
        p1_txt = input("Digite o polo 1 (a+b*j): ") 
        p2_txt = input("Digite o polo 2 (a-b*j): ") 

        polo_1 = sp.parse_expr(p1_txt, local_dict={'j': sp.I})
        polo_2 = sp.parse_expr(p2_txt, local_dict={'j': sp.I})

        sigma = - polo_1.real()
        w_d = polo_1.imag()
    return polo_1, polo_2, w_d, sigma


In [42]:
def angulo(v):
    """Retorna o argumento de um número complexo em graus."""
    return float(sp.deg(sp.arg(v)))


In [43]:
def calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, pid):
    if(pid):
        div = 2
    else:
        div = 1

    ponto_si = polo_1

    diferenca_polos = np.empty((0,2))
    diferenca_zeros = np.empty((0,2))

    #print("Diferença entre o ponto e os polos: ")
    for polo in polos:
        diferenca = ponto_si - polo
        modulo = abs(diferenca)
        fase = math.degrees(cmath.phase(diferenca))
        nova_linha = [[modulo.round(4), fase]]
        diferenca_polos = np.append(diferenca_polos, nova_linha, axis=0)
        #print(f"{polo}: {nova_linha}")
    #print

    #print("Diferença entre o ponto e os zeros: ")
    for zero in zeros:
        diferenca = ponto_si - zero
        modulo = abs(diferenca)
        fase = math.degrees(cmath.phase(diferenca))
        nova_linha = [[modulo.round(4), fase]]
        diferenca_zeros = np.append(diferenca_zeros, nova_linha, axis=0)
        #print(f"{zeros}: {nova_linha}")

    produto_polos = 1
    produto_zeros = 1
    theta = 0

    for linha in diferenca_polos:
        produto_polos = produto_polos * linha[0]
        theta = theta - linha[1]

    for linha in diferenca_zeros:
        produto_zeros = produto_zeros * linha[0]
        theta = theta + linha[1]

    M = ganho * produto_zeros/produto_polos

    # Tratamento para o angulo ser positivo
    theta = theta % 360
    theta = theta / div # Divide por 2 se for PID

    angulo_desejado = 180 - theta
    print(f"Angulo = {angulo_desejado:.2f}°")

    freq = ft.eval_frequency(polo_1)
    M = sp.Abs(freq)
    theta = cmath.phase(freq) * 180/np.pi # a fase deve ser em graus
    # Tratamento para o angulo ser positivo
    theta = theta % 360

    angulo_desejado = 180 - theta
    print(f"Ângulo entre polo desejado e o zero = {angulo_desejado}")

    z_c = sigma + (w_d / math.tan(math.radians(angulo_desejado)))
    print(f"Zero do controlador: {z_c:.2f} + 0*j")

    Gc = TransferFunction(s + z_c, 1, s)
    ft_c = Series(ft, Gc).doit()

    freq = ft_c.eval_frequency(ponto_si)
    M = sp.Abs(freq)
    Kc = 1/M
    print(f"Kc = {Kc}\n")
    return z_c, Kc

## Traduzir as especificações de desempenho

In [44]:
PI, PD, PID = definir_controlador()

Controlador PID


In [47]:
metodo_1, metodo_2, metodo_3 = definir_problema()

Digite 1 para Mp e Epsilon.
Digite 2 para fator de amortecimento e frequência natural de amortecimento.
Digite 3 para localização de polos de malha


In [52]:
polo_1, polo_2, w_d, sigma = definir_polos_dominantes(metodo_1, metodo_2, metodo_3)

AttributeError: 'Add' object has no attribute 'real'

In [ ]:
print("Forneça os coeficientes da função G(s)...")
numerador_G = polinomio(input("Digite os coeficientes do numerador: "))
denominador_G = polinomio(input("Digite os coeficientes do denominador: "))

numerador_G = sp.parse_expr(numerador_G, local_dict={'s': s})
denominador_G = sp.parse_expr(denominador_G, local_dict={'s': s})

G = TransferFunction(numerador_G, denominador_G, s)
G

Forneça os coeficientes da função G(s)...


TransferFunction(4*s + 16, s**3 + 4*s**2 + 4*s, s)

In [ ]:
print("Forneça os coeficientes da função H(s)...")
numerador_H = polinomio(input("Digite os coeficientes do numerador: "))
denominador_H = polinomio(input("Digite os coeficientes do denominador: "))

numerador_H = sp.parse_expr(numerador_H, local_dict={'s': s})
denominador_H = sp.parse_expr(denominador_H, local_dict={'s': s})

H = TransferFunction(numerador_H, denominador_H, s)
H

Forneça os coeficientes da função H(s)...


TransferFunction(1, 1, s)

In [ ]:
ft = Series(G, H).doit()
ft

TransferFunction(4*s + 16, s**3 + 4*s**2 + 4*s, s)

In [ ]:
zeros, polos = pole_zero_numerical_data(ft)
print(f"Polos: {polos}")
print(f"Zeros: {zeros}")

Polos: [-2.0, -2.0, 0.0]
Zeros: [-4.0]


In [ ]:
# Verifica se a função de transferência tem naturalmente um ganho
num = ft.num
den = ft.den

ganho = Poly(num, s).LC() / Poly(den, s).LC()
print(f"Ganho = {ganho}")

Ganho = 4


## PD

In [ ]:
if(PD):
    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID)

    Kd = Kc
    Kp = z_c * Kc
    print(f"Kp = {Kp} \nKd = {Kd}")
    

Angulo = 7.37°
Ângulo entre polo desejado e o zero = 7.371373875171116
Zero do controlador: 8.66 + 0*j
Kc = 0.0304577480820625

Kp = 0.263757278455857 
Kd = 0.0304577480820625


## PI

In [ ]:
if(PI):
    polos.append(0 + 0*I)
    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID)

    Kp = Kc
    Ki = z_c * Kc
    print(f"Kp = {Kp} \nKi = {Ki}")
    

In [ ]:
if(PID):
    polos.append(0 + 0*I)
    z_c, Kc = calcular_zc(ft, polo_1, polos, zeros, ganho, w_d, sigma, PID)

    Kp = 2*Kc*z_c
    Ki = z_c**2
    Kd = Kc
    print(f"Kp = {Kp} \nKi = {Ki} \nKd = {Kd}")
    